# 1D Matérn smooth misspecification diagnostic

This notebook isolates the pure-space issue in one dimension.

It simulates exact 1D Gaussian-process data with true Matérn smooth
`nu ∈ {0.3, 0.5, 1.5}`, then fits models with fitted smooth
`nu_fit ∈ {0.3, 0.5, 1.5}` across resolution thinning levels.

Fit families:

- `full_nugget_free`: estimate `sigmasq`, `range`, `nugget`
- `full_nugget0`: estimate `sigmasq`, `range`, fix `nugget=0`
- `sigma_only_nugget_free`: estimate `sigmasq`, fix `range/nugget`
- `range_only_nugget_free`: estimate `range`, fix `sigmasq/nugget`
- `nugget_only`: estimate `nugget`, fix `sigmasq/range`
- `sigma_only_nugget0`: estimate `sigmasq`, fix `range`, `nugget=0`
- `range_only_nugget0`: estimate `range`, fix `sigmasq`, `nugget=0`


In [ ]:
import itertools
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import gamma as scipy_gamma, kv as scipy_kv
from scipy.linalg import cho_factor, cho_solve

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROUND_DECIMALS = 4

PURE_SPACE_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space')
OUT_DIR = PURE_SPACE_DIR / 'log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'sim_1d_matern_smooth_misspec_r0p2_050826'

def round_df(df, ndigits=ROUND_DECIMALS):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(ndigits)
    return out


In [ ]:
# Main experiment config.
TRUE_SMOOTHS = [0.3, 0.5, 1.5]
FIT_SMOOTHS = [0.3, 0.5, 1.5]
RESOLUTION_STRIDES = [8, 4, 2, 1]

# Keep this modest: exact likelihood does repeated Cholesky factorizations.
N_FULL = 256
N_REPS = 5
SIM_SEED = 20260508

TRUE_PARAMS = {
    'sigmasq': 12.0,
    'range': 0.20,
    'nugget': 2.0,
}
JITTER = 1e-7

FIT_TYPES = {
    'full_nugget_free': {
        'free': ['sigmasq', 'range', 'nugget'],
        'fixed': {},
    },
    'full_nugget0': {
        'free': ['sigmasq', 'range'],
        'fixed': {'nugget': 0.0},
    },
    'sigma_only_nugget_free': {
        'free': ['sigmasq'],
        'fixed': {'range': TRUE_PARAMS['range'], 'nugget': TRUE_PARAMS['nugget']},
    },
    'range_only_nugget_free': {
        'free': ['range'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'nugget': TRUE_PARAMS['nugget']},
    },
    'nugget_only': {
        'free': ['nugget'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'range': TRUE_PARAMS['range']},
    },
    'sigma_only_nugget0': {
        'free': ['sigmasq'],
        'fixed': {'range': TRUE_PARAMS['range'], 'nugget': 0.0},
    },
    'range_only_nugget0': {
        'free': ['range'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'nugget': 0.0},
    },
}

PARAM_BOUNDS = {
    'sigmasq': (1e-4, 200.0),
    'range': (0.005, 0.8),
    'nugget': (1e-8, 100.0),
}

INIT_PARAMS = {
    'sigmasq': 10.0,
    'range': 0.20,
    'nugget': 1.0,
}

print('true smooths:', TRUE_SMOOTHS)
print('fit smooths:', FIT_SMOOTHS)
print('resolutions:', {f'x{s}': int(np.ceil(N_FULL / s)) for s in RESOLUTION_STRIDES})
print('fit types:', list(FIT_TYPES))


In [ ]:
def resolution_label(stride):
    return f'x{int(stride)}'


def matern_corr_1d(dist_over_range, smooth):
    r = np.asarray(dist_over_range, dtype=float)
    nu = float(smooth)
    if np.isclose(nu, 0.5):
        return np.exp(-r)
    z = np.sqrt(2.0 * nu) * r
    safe = np.where(z < 1e-12, 1e-12, z)
    out = (2.0 ** (1.0 - nu) / scipy_gamma(nu)) * (safe ** nu) * scipy_kv(nu, safe)
    out = np.where(z < 1e-12, 1.0, out)
    return np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)


def cov_1d(x, smooth, sigmasq, range_, nugget=0.0):
    x = np.asarray(x, dtype=float)
    d = np.abs(x[:, None] - x[None, :])
    K = float(sigmasq) * matern_corr_1d(d / float(range_), smooth)
    if float(nugget) > 0:
        K = K + float(nugget) * np.eye(len(x))
    return K


def simulate_exact_1d(rng, x, smooth, params):
    K = cov_1d(x, smooth, params['sigmasq'], params['range'], params['nugget'])
    K = K + JITTER * np.eye(len(x))
    L = np.linalg.cholesky(K)
    return L @ rng.standard_normal(len(x))


def neg_loglik(y, x, smooth, sigmasq, range_, nugget):
    n = len(y)
    K = cov_1d(x, smooth, sigmasq, range_, nugget) + JITTER * np.eye(n)
    try:
        cf = cho_factor(K, lower=True, check_finite=False)
        alpha = cho_solve(cf, y, check_finite=False)
        logdet = 2.0 * np.sum(np.log(np.diag(cf[0])))
        return 0.5 * (logdet + float(y @ alpha) + n * np.log(2.0 * np.pi))
    except Exception:
        return np.inf


def unpack_params(theta, free_names, fixed):
    vals = dict(fixed)
    for name, logv in zip(free_names, theta):
        vals[name] = float(np.exp(logv))
    for name in ['sigmasq', 'range', 'nugget']:
        vals.setdefault(name, TRUE_PARAMS[name])
    return vals


def fit_exact_1d(y, x, fit_smooth, fit_type):
    spec = FIT_TYPES[fit_type]
    free = spec['free']
    fixed = dict(spec['fixed'])
    theta0 = np.array([np.log(INIT_PARAMS[name]) for name in free], dtype=float)
    bounds = [(np.log(PARAM_BOUNDS[name][0]), np.log(PARAM_BOUNDS[name][1])) for name in free]

    def obj(theta):
        p = unpack_params(theta, free, fixed)
        return neg_loglik(y, x, fit_smooth, p['sigmasq'], p['range'], p['nugget'])

    if free:
        opt = minimize(obj, theta0, method='L-BFGS-B', bounds=bounds, options={'maxiter': 80, 'ftol': 1e-8})
        theta_hat = opt.x
        loss = float(opt.fun)
        success = bool(opt.success)
        n_iter = int(getattr(opt, 'nit', -1))
    else:
        theta_hat = np.array([])
        loss = float(obj(theta_hat))
        success = True
        n_iter = 0

    est = unpack_params(theta_hat, free, fixed)
    return est, loss / len(y), success, n_iter


In [ ]:
# Simulate once at full resolution for each true smooth and replicate.
x_full = np.linspace(0.0, 1.0, N_FULL)
rng = np.random.default_rng(SIM_SEED)

sim_rows = []
sim_data = {}
for true_smooth in TRUE_SMOOTHS:
    for rep in range(N_REPS):
        y = simulate_exact_1d(rng, x_full, true_smooth, TRUE_PARAMS)
        key = (float(true_smooth), int(rep))
        sim_data[key] = y
        sim_rows.append({
            'true_smooth': true_smooth,
            'rep': rep,
            'y_mean': float(np.mean(y)),
            'y_var': float(np.var(y, ddof=1)),
        })

sim_df = pd.DataFrame(sim_rows)
display(round_df(sim_df))


In [ ]:
# Run all exact likelihood fits.
results = []
t0_all = time.time()

for true_smooth in TRUE_SMOOTHS:
    for fit_smooth in FIT_SMOOTHS:
        for rep in range(N_REPS):
            y_full = sim_data[(float(true_smooth), int(rep))]
            for stride in RESOLUTION_STRIDES:
                idx = np.arange(0, N_FULL, int(stride), dtype=int)
                x = x_full[idx]
                y = y_full[idx]
                for fit_type in FIT_TYPES:
                    t0 = time.time()
                    est, loss_per_obs, success, n_iter = fit_exact_1d(y, x, fit_smooth, fit_type)
                    elapsed = time.time() - t0
                    row = {
                        'true_smooth': float(true_smooth),
                        'fit_smooth': float(fit_smooth),
                        'rep': int(rep),
                        'resolution_stride': int(stride),
                        'resolution_label': resolution_label(stride),
                        'fit_type': fit_type,
                        'n': int(len(y)),
                        'loss_per_obs': float(loss_per_obs),
                        'success': success,
                        'n_iter': n_iter,
                        'fit_s': float(elapsed),
                        'est_sigmasq': est['sigmasq'],
                        'est_range': est['range'],
                        'est_nugget': est['nugget'],
                        'true_sigmasq': TRUE_PARAMS['sigmasq'],
                        'true_range': TRUE_PARAMS['range'],
                        'true_nugget': TRUE_PARAMS['nugget'],
                    }
                    results.append(row)
        tmp = pd.DataFrame(results)
        tmp_path = OUT_DIR / f'{OUT_PREFIX}_fits_partial.csv'
        round_df(tmp).to_csv(tmp_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
        print(f'done true={true_smooth}, fit={fit_smooth}; rows={len(results)}; elapsed={time.time() - t0_all:.1f}s')

fit_df = pd.DataFrame(results)
fit_path = OUT_DIR / f'{OUT_PREFIX}_fits.csv'
round_df(fit_df).to_csv(fit_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved fits:', fit_path)
display(round_df(fit_df.head(20)))


In [ ]:
# Summary by true smooth, fitted smooth, resolution, and fit type.
summary_df = (
    fit_df
    .groupby(['true_smooth', 'fit_smooth', 'resolution_stride', 'resolution_label', 'fit_type'], observed=False)
    .agg(
        n_reps=('rep', 'nunique'),
        loss_mean=('loss_per_obs', 'mean'),
        sigmasq_mean=('est_sigmasq', 'mean'),
        sigmasq_sd=('est_sigmasq', 'std'),
        range_mean=('est_range', 'mean'),
        range_sd=('est_range', 'std'),
        nugget_mean=('est_nugget', 'mean'),
        nugget_sd=('est_nugget', 'std'),
        success_rate=('success', 'mean'),
        fit_s_mean=('fit_s', 'mean'),
    )
    .reset_index()
)
summary_path = OUT_DIR / f'{OUT_PREFIX}_summary.csv'
round_df(summary_df).to_csv(summary_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved summary:', summary_path)
display(round_df(summary_df.head(30)))


In [ ]:
# Plot helpers.
PARAM_INFO = {
    'sigmasq': ('est_sigmasq', TRUE_PARAMS['sigmasq']),
    'range': ('est_range', TRUE_PARAMS['range']),
    'nugget': ('est_nugget', TRUE_PARAMS['nugget']),
}

def plot_param_grid(fit_type, parameter, y_shared=True):
    y_col, true_value = PARAM_INFO[parameter]
    sub = fit_df[fit_df['fit_type'] == fit_type].copy()
    if sub.empty:
        raise ValueError(f'No rows for fit_type={fit_type}')
    sub['resolution_label'] = pd.Categorical(
        sub['resolution_label'], categories=[resolution_label(s) for s in RESOLUTION_STRIDES], ordered=True
    )
    stats = (
        sub
        .groupby(['true_smooth', 'fit_smooth', 'resolution_label'], observed=False)
        .agg(mean=(y_col, 'mean'), sd=(y_col, 'std'))
        .reset_index()
        .sort_values(['true_smooth', 'fit_smooth', 'resolution_label'])
    )

    fig, axes = plt.subplots(len(TRUE_SMOOTHS), len(FIT_SMOOTHS), figsize=(14, 10), constrained_layout=True, sharex=True, sharey=y_shared)
    for i, true_smooth in enumerate(TRUE_SMOOTHS):
        for j, fit_smooth in enumerate(FIT_SMOOTHS):
            ax = axes[i, j]
            s = stats[(stats['true_smooth'] == true_smooth) & (stats['fit_smooth'] == fit_smooth)]
            ax.errorbar(s['resolution_label'].astype(str), s['mean'], yerr=s['sd'], marker='o', capsize=3)
            ax.axhline(true_value, color='black', linestyle=':', linewidth=1.2, label='truth' if (i == 0 and j == 0) else None)
            ax.set_title(f'true nu={true_smooth}, fit nu={fit_smooth}')
            ax.grid(True, alpha=0.25)
            if j == 0:
                ax.set_ylabel(parameter)
            if i == len(TRUE_SMOOTHS) - 1:
                ax.set_xlabel('resolution thinning')
    if axes[0, 0].get_legend_handles_labels()[0]:
        axes[0, 0].legend(fontsize=8)
    fig.suptitle(f'1D exact GP: {fit_type}, parameter={parameter}', fontsize=14)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_{parameter}_grid.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()


def plot_loss_grid(fit_type, y_shared=True):
    sub = fit_df[fit_df['fit_type'] == fit_type].copy()
    sub['resolution_label'] = pd.Categorical(
        sub['resolution_label'], categories=[resolution_label(s) for s in RESOLUTION_STRIDES], ordered=True
    )
    stats = (
        sub
        .groupby(['true_smooth', 'fit_smooth', 'resolution_label'], observed=False)
        .agg(mean=('loss_per_obs', 'mean'), sd=('loss_per_obs', 'std'))
        .reset_index()
        .sort_values(['true_smooth', 'fit_smooth', 'resolution_label'])
    )
    fig, axes = plt.subplots(len(TRUE_SMOOTHS), len(FIT_SMOOTHS), figsize=(14, 10), constrained_layout=True, sharex=True, sharey=y_shared)
    for i, true_smooth in enumerate(TRUE_SMOOTHS):
        for j, fit_smooth in enumerate(FIT_SMOOTHS):
            ax = axes[i, j]
            s = stats[(stats['true_smooth'] == true_smooth) & (stats['fit_smooth'] == fit_smooth)]
            ax.errorbar(s['resolution_label'].astype(str), s['mean'], yerr=s['sd'], marker='o', capsize=3)
            ax.set_title(f'true nu={true_smooth}, fit nu={fit_smooth}')
            ax.grid(True, alpha=0.25)
            if j == 0:
                ax.set_ylabel('negative loglik / obs')
            if i == len(TRUE_SMOOTHS) - 1:
                ax.set_xlabel('resolution thinning')
    fig.suptitle(f'1D exact GP loss: {fit_type}', fontsize=14)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_loss_grid.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()


In [ ]:
# Core plots: full fits with and without nugget.
for fit_type in ['full_nugget_free', 'full_nugget0']:
    for parameter in ['sigmasq', 'range', 'nugget']:
        plot_param_grid(fit_type, parameter)
    plot_loss_grid(fit_type)


In [ ]:
# Fixed-parameter profile plots.
for fit_type, parameter in [
    ('sigma_only_nugget_free', 'sigmasq'),
    ('range_only_nugget_free', 'range'),
    ('nugget_only', 'nugget'),
    ('sigma_only_nugget0', 'sigmasq'),
    ('range_only_nugget0', 'range'),
]:
    plot_param_grid(fit_type, parameter)
    plot_loss_grid(fit_type)


In [ ]:
# Heatmap at a selected resolution: quick view of smooth misspecification bias.
HEATMAP_RESOLUTION = 'x1'

def plot_heatmap_at_resolution(fit_type, parameter, resolution_label_=HEATMAP_RESOLUTION):
    y_col, true_value = PARAM_INFO[parameter]
    sub = fit_df[(fit_df['fit_type'] == fit_type) & (fit_df['resolution_label'] == resolution_label_)].copy()
    mat = (
        sub
        .groupby(['true_smooth', 'fit_smooth'], observed=False)[y_col]
        .mean()
        .unstack('fit_smooth')
        .reindex(index=TRUE_SMOOTHS, columns=FIT_SMOOTHS)
    )
    bias = mat - true_value

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    im0 = axes[0].imshow(mat.to_numpy(), aspect='auto')
    axes[0].set_title(f'{parameter} estimate')
    im1 = axes[1].imshow(bias.to_numpy(), aspect='auto', cmap='coolwarm')
    axes[1].set_title(f'{parameter} bias')
    for ax in axes:
        ax.set_xticks(range(len(FIT_SMOOTHS)), FIT_SMOOTHS)
        ax.set_yticks(range(len(TRUE_SMOOTHS)), TRUE_SMOOTHS)
        ax.set_xlabel('fit smooth')
        ax.set_ylabel('true smooth')
    for ax, arr in [(axes[0], mat.to_numpy()), (axes[1], bias.to_numpy())]:
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                ax.text(j, i, f'{arr[i, j]:.3g}', ha='center', va='center', color='white')
    fig.colorbar(im0, ax=axes[0], shrink=0.8)
    fig.colorbar(im1, ax=axes[1], shrink=0.8)
    fig.suptitle(f'{fit_type}, {resolution_label_}', fontsize=13)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_{parameter}_{resolution_label_}_heatmap.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()


for fit_type in ['full_nugget_free', 'full_nugget0', 'sigma_only_nugget_free', 'range_only_nugget_free', 'nugget_only']:
    for parameter in ['sigmasq', 'range', 'nugget']:
        if parameter == 'nugget' and fit_type in ['full_nugget0', 'range_only_nugget0', 'sigma_only_nugget0']:
            continue
        if parameter == 'range' and fit_type in ['sigma_only_nugget_free', 'sigma_only_nugget0', 'nugget_only']:
            continue
        if parameter == 'sigmasq' and fit_type in ['range_only_nugget_free', 'range_only_nugget0', 'nugget_only']:
            continue
        plot_heatmap_at_resolution(fit_type, parameter)


In [ ]:
# Optional: 1D spectrum check for one selected scenario.
SPECTRUM_TRUE_SMOOTH = 0.3
SPECTRUM_FIT_SMOOTH = 0.3
SPECTRUM_REP = 0
SPECTRUM_FIT_TYPE = 'full_nugget_free'

def periodogram_1d(y):
    z = y - np.mean(y)
    win = np.hanning(len(z))
    p = np.abs(np.fft.rfft(z * win)) ** 2 / max(np.sum(win ** 2), 1e-12)
    f = np.fft.rfftfreq(len(z), d=1.0 / (len(z) - 1))
    return f[1:], p[1:]

def matern_spectrum_1d_shape(freq, smooth, sigmasq, range_, nugget):
    omega = 2.0 * np.pi * np.asarray(freq, dtype=float)
    nu = float(smooth)
    alpha = 2.0 * nu / max(float(range_) ** 2, 1e-12)
    return float(sigmasq) * (alpha + omega ** 2) ** (-(nu + 0.5)) + max(float(nugget), 0.0)

fig, axes = plt.subplots(1, len(RESOLUTION_STRIDES), figsize=(16, 3.5), constrained_layout=True, sharey=True)
for ax, stride in zip(axes, RESOLUTION_STRIDES):
    idx = np.arange(0, N_FULL, int(stride), dtype=int)
    y = sim_data[(float(SPECTRUM_TRUE_SMOOTH), int(SPECTRUM_REP))][idx]
    f, p = periodogram_1d(y)
    fit_row = fit_df[
        (fit_df['true_smooth'] == SPECTRUM_TRUE_SMOOTH)
        & (fit_df['fit_smooth'] == SPECTRUM_FIT_SMOOTH)
        & (fit_df['rep'] == SPECTRUM_REP)
        & (fit_df['resolution_stride'] == stride)
        & (fit_df['fit_type'] == SPECTRUM_FIT_TYPE)
    ].iloc[0]
    th = matern_spectrum_1d_shape(f, SPECTRUM_FIT_SMOOTH, fit_row.est_sigmasq, fit_row.est_range, fit_row.est_nugget)
    good = (p > 0) & (th > 0)
    scale = np.exp(np.mean(np.log(p[good]) - np.log(th[good]))) if good.any() else 1.0
    ax.plot(f, p, color='black', linewidth=1.5, label='data periodogram')
    ax.plot(f, th * scale, color='tab:red', linestyle='--', linewidth=1.5, label='fitted theory')
    ax.set_yscale('log')
    ax.set_title(resolution_label(stride))
    ax.set_xlabel('cycles per unit interval')
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('spectrum')
axes[0].legend(fontsize=8)
fig.suptitle(f'1D spectrum: true nu={SPECTRUM_TRUE_SMOOTH}, fit nu={SPECTRUM_FIT_SMOOTH}, {SPECTRUM_FIT_TYPE}', fontsize=13)
out = OUT_DIR / f'{OUT_PREFIX}_selected_spectrum.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
print('Saved:', out)
plt.show()
